In [0]:
from pyspark.sql.functions import col, when, year, month

# 1. Read the cleansed data from the Silver table
df_orders_silver_full = spark.table("workspace.my_database.orders_silver")

# 2. Chain all transformations together (Removed 'if' conditions for strict ETL validation)
df_orders_gold = df_orders_silver_full \
    .withColumn(
        "order_year", year(col("order_date"))
    ) \
    .withColumn(
        "order_month", month(col("order_date"))
    ) \
    .withColumn(
        "order_size",
        when(col("total_amount") < 50, "Small")
        .when((col("total_amount") >= 50) & (col("total_amount") <= 200), "Medium")
        .otherwise("Large")
    ) \
    .withColumn(
        "is_priority_order",
        when(col("status") == "Urgent", True).otherwise(False)
    )

# 3. Save the enriched dataset as the new Gold table
df_orders_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.my_database.orders_gold")
print("Orders Gold Layer has been enriched perfectly and saved successfully!")

# 4. Display the data to verify the new columns
display(df_orders_gold)